In [ ]:
import pandas as pd

# Specify the path to your Excel file
excel_file_path = 'GBM_Bibliography_Abstracts.xlsx'

# Specify the sheet name
sheet_name = 'Bibliography'

# Read a specific sheet into a DataFrame
df = pd.read_excel(excel_file_path, sheet_name=sheet_name)

# # Create a sample DataFrame
# data = {'Name': ['Alice', 'Bob', 'Charlie'],
#         'Age': [25, 30, 35],
#         'City': ['New York', 'San Francisco', 'Los Angeles']}

# Display the DataFrame
# print(df)

In [ ]:
import numpy as np
import torch
# from os import listdir, makedirs
# from os.path import isfile, join, splitext, exists

from transformers import BertModel, BertTokenizer
model = BertModel.from_pretrained('bert-base-uncased',
           output_hidden_states = True,)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# Extract features
def extract_features(data_frame):
    
    feature_vectors = []
    IDs = []
    
    # Iterate through rows using iterrows()
    for index, row in data_frame.iterrows():
        ID = row['ID']
        text = row['Title'] + ' ' + row['Abstract']
        
        # print(f"Index: {index}, ID: {ID}, Text: {text}")
        
        
        if index % 10 == 0:
            print("Processing sample %d..." % (index))
            
        tokens = tokenizer(text, truncation=False, return_tensors='pt')        
        if tokens.input_ids.shape[1] > 512:
            print("Too many tokens for %s: %d" % (ID, tokens.input_ids.shape[1]))
        
        tokens = tokenizer(text, truncation=True, max_length=512, return_tensors='pt')
        outputs = model(**tokens, output_hidden_states=True)
        
        last_hidden_states = outputs.hidden_states[-1] # last layer hidden states
        
        cls_token = torch.mean(last_hidden_states[0], dim=0)#last_hidden_states[0,0,:]
        
        feature_vector = np.squeeze(cls_token.detach().numpy())
        feature_vectors += [feature_vector]
        IDs += [ID]
        
    return feature_vectors, IDs
        
feature_vectors, IDs = extract_features(df)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
def extract_tfidf(df):
    df['text'] = df['Title'] + ' ' + df['Abstract']
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')
    tfidf = tfidf_vectorizer.fit_transform(df['text'])
    return tfidf.toarray(), list(df['ID'])
# feature_vectors, IDs = extract_tfidf(df)

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, median, inconsistent, maxinconsts
from scipy.spatial.distance import pdist
import numpy as np
print(len(IDs))
# Generate some random data for demonstration
np.random.seed(837)

D = pdist(feature_vectors, 'euclidean')

# Perform hierarchical clustering using Ward linkage
linkage_matrix = linkage(feature_vectors, method='ward')
R = inconsistent(linkage_matrix)

max_inco = maxinconsts(linkage_matrix, R)
# print(max_inco)
print("Max inco: %f" % (np.mean(max_inco)))

# Display the dendrogram
plt.figure(figsize=(10, 14))
dendrogram(linkage_matrix, labels=IDs, color_threshold=5.2, orientation = 'left')
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.ylabel('Data Points')
# plt.xlim(1.4, 0.8)
plt.xlabel('Distance')
plt.savefig('GBM_dendrogram_BERT_5.2.pdf', bbox_inches='tight', pad_inches=0.2)
plt.show()